# 第4日 ノートブック01：データセットと注釈

# Day 4 Notebook 01: Dataset Preparation and Annotation

---

## 学習目標 / Learning objectives

| 日本語 | English |
| --- | --- |
| 1. 教師あり学習に必要なものを説明できる | 1. Explain what supervised learning needs |
| 2. YOLO 形式の正解データを読んで意味を説明できる | 2. Read YOLO-format labels and explain them |
| 3. 正規化座標を自分で計算できる | 3. Compute normalised coordinates yourself |
| 4. 学習・検証・試験データの役割と、その分離の理由を説明できる | 4. Explain train/val/test roles and why they must be separated |
| 5. データ漏れ（data leakage）がなぜ問題かを説明できる | 5. Explain why data leakage is a problem |

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 80 分 / 80 minutes |
| 難易度 / Difficulty | 初級 / Beginner |
| 前提条件 / Prerequisites | 第3日を終えていること。 / You have finished Day 3. |
| 必要なライブラリ / Required libraries | numpy, matplotlib, opencv-python, pillow, pandas |
| 使用カーネル / Jupyter kernel | aivision |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第4日 / Day 4](../../COURSE_SYLLABUS.md#day4) |

### 提出物 / Expected deliverables

| 日本語 | English |
| --- | --- |
| 自分のデータセットの点検結果 | Your dataset inspection results |
| 手計算した YOLO 形式の座標 | A hand-computed YOLO label |

## このノートブックの位置 / Where this notebook sits

| 項目 / Item | 内容 / Details |
| --- | --- |
| 日 / Day | 第4日 / Day 4 |
| この日の順番 / Position today | 1 / 3（この日のノートブックのうち）/ of today's notebooks |
| 所要時間 / Duration | 80 分 / 80 minutes |
| 前提 / Prerequisites | 第3日 / Day 3 |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第4日 / Day 4](../../COURSE_SYLLABUS.md#day4) |

1 日の時間割は `COURSE_SYLLABUS.md` に 1 つだけあります。このノートブックはその一部で、
自分の時間割を持ちません。時刻が分からなくなったら、まずそこを見てください。

The day's timetable exists once, in `COURSE_SYLLABUS.md`. This notebook is one part of it and
does not keep its own schedule; look there first when you are unsure of the time.


---

## Part 0　準備 / Preparation

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd().resolve()
while not ((COURSE_ROOT / "assets").is_dir() and (COURSE_ROOT / "notebooks").is_dir()):
    if COURSE_ROOT.parent == COURSE_ROOT:
        raise FileNotFoundError(
            "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
            "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
        )
    COURSE_ROOT = COURSE_ROOT.parent

ASSETS = COURSE_ROOT / "assets"
sys.path.insert(0, str(ASSETS / "helpers"))
import aivision

print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("cv2", "opencv-python"),
    ("pandas", "pandas"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
FONT_FAMILY = aivision.configure_japanese_font()
aivision.set_plot_style()

if FONT_FAMILY is None:
    print("日本語フォントが見つかりません。図のタイトルは英字になります。")
    print("No Japanese font found; figure titles fall back to English.")
else:
    print("図の日本語フォント / figure font:", FONT_FAMILY)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

def show(images, titles=None, figsize=(14, 5)):
    """画像を並べて表示する / display images side by side."""
    if not isinstance(images, (list, tuple)):
        images = [images]
    if titles is None:
        titles = [""] * len(images)
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image)
        ax.set_title(title, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()


def detections_to_frame(result):
    """検出結果を pandas の表にする / convert detections into a DataFrame.

    ``result.boxes.data`` の 6 列は
    (x1, y1, x2, y2, confidence, class_id) です。
    """
    columns = ["class_id", "name", "confidence", "x1", "y1", "x2", "y2"]
    boxes = result.boxes
    if len(boxes) == 0:
        return pd.DataFrame(columns=columns)

    data = boxes.data.cpu().numpy()
    return pd.DataFrame({
        "class_id": data[:, 5].astype(int),
        "name": [result.names[int(class_id)] for class_id in data[:, 5]],
        "confidence": data[:, 4].round(4),
        "x1": data[:, 0].round(2),
        "y1": data[:, 1].round(2),
        "x2": data[:, 2].round(2),
        "y2": data[:, 3].round(2),
    })

#### このデータセットについて
この研修で使うデータセットは **すべて合成画像** です。
Python で描いた「PVC シートらしい」画像で、実際の工場画像ではありません。

だからこそ、**正解の位置が正確** です。
実際の不良品では「この傷はどこまでか」を人が決める必要があり、意見が分かれます。
ここでは、描画した位置がそのまま正解になります。

#### About this dataset
Every image in this course dataset is **synthetic**, drawn by Python to look like a PVC
sheet. None of them come from the factory.

That is exactly why the **ground truth is exact**: for real defects, humans must decide where a
scratch begins and ends, and they disagree. Here the drawn position *is* the truth.

---

## Part 1　教師あり学習 / Supervised learning

### 実行結果の例 / Expected output

次のような結果になります（数値は環境によって多少変わります）。

```text
クラス / classes: ['black_spot', 'scratch', 'streak']
画像サイズ / image size: 320

train     64            52                12  {'black_spot': 18, 'scratch': 26, 'streak': 26}
val       16            13                 3     {'black_spot': 5, 'scratch': 7, 'streak': 6}
test      16            13                 3     {'black_spot': 5, 'scratch': 7, 'streak': 6}

重複画像（分割間）/ duplicate images across splits: 0
```

You should see something like the following (exact numbers vary slightly between machines).

```text
classes: ['black_spot', 'scratch', 'streak']
image size: 320

split   images  defects  background  instances
train       64       52          12  black_spot 18 / scratch 26 / streak 26
val         16       13           3  black_spot  5 / scratch  7 / streak  6
test        16       13           3  black_spot  5 / scratch  7 / streak  6

duplicate images across splits: 0
```

#### 「答え付きの例題」から学ぶ
**教師あり学習** は、**入力と正解のペア** を大量に見せて学ばせます。

```text
入力：シートの画像
正解：この位置に黒点がある（四角とクラス名）
```

この「正解」を **注釈（アノテーション）** と呼びます。

| 用語 | 意味 |
|---|---|
| 注釈 / アノテーション | 画像に対する正解情報 |
| バウンディングボックス | 欠陥を囲む四角 |
| クラスラベル | 欠陥の種類（黒点・傷・スジ） |
| 正解データ | 注釈をファイルにしたもの |

注釈を作る作業は **人間の仕事** です。ここに時間がかかり、
間違えると AI も間違えます。**AI の性能の上限は、注釈の質で決まります。**

#### Learning from examples with answers
**Supervised learning** learns from many **input / correct-answer pairs**.

```text
input : an image of the sheet
answer: there is a black spot here (box + class name)
```

That answer is called an **annotation**.

| Term | Meaning |
|---|---|
| annotation | the correct answer for an image |
| bounding box | the rectangle around a defect |
| class label | the defect type (spot / scratch / streak) |
| ground truth | annotations stored as files |

Creating annotations is **human work**. It is slow, and mistakes become the model's mistakes.
**The quality of the annotations sets the ceiling on model quality.**

---

## Part 2　データセットの構造 / Dataset structure

#### YOLO データセットの並び
YOLO のデータセットは、決まった並びにします。

```text
assets/yolo_dataset/
    images/
        train/  train_0000.png ...
        val/    val_0000.png ...
        test/   test_0000.png ...
    labels/
        train/  train_0000.txt ...   <- 画像と同じ名前の .txt
        val/    ...
        test/   ...
    data.yaml
```

大切なのは、**画像とラベルが同じファイル名** であることです
（拡張子だけ違う）。`train_0000.png` には `train_0000.txt` が対応します。

`data.yaml` には、クラス名とフォルダの場所を書きます。

#### The YOLO dataset layout
A YOLO dataset uses a fixed layout:

```text
assets/yolo_dataset/
    images/train/, images/val/, images/test/
    labels/train/, labels/val/, labels/test/
    data.yaml
```

The key rule is that **an image and its label share the same file name** (only the extension
differs): `train_0000.png` pairs with `train_0000.txt`.

`data.yaml` declares the class names and where the folders are.

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import json
import pandas as pd

dataset_dir = ASSETS / "yolo_dataset"
data_yaml = dataset_dir / "data.yaml"
summary_path = dataset_dir / "dataset_summary.json"

print("dataset folder :", dataset_dir)
print("data.yaml      :", data_yaml.is_file())
print()
print("--- data.yaml の中身 / contents ---")
print(data_yaml.read_text(encoding="utf-8") if data_yaml.is_file() else "NOT FOUND")

#### `data.yaml` の意味

| 行 | 意味 |
|---|---|
| `path` | データセットのルートフォルダ |
| `train` / `val` / `test` | それぞれの画像フォルダ（`path` からの相対） |
| `names` | クラス番号 → クラス名 |

> `path` は絶対パスで書かれています。
> **フォルダを移動したら、このファイルを書き直す必要があります。**
> その手順は次のノートブック（04_02）で行います。

#### What `data.yaml` means
| Line | Meaning |
|---|---|
| `path` | dataset root folder |
| `train` / `val` / `test` | image folders, relative to `path` |
| `names` | class index to class name |

`path` is written as an absolute path, so **if you move the folder you must rewrite this file**.
The training notebook (04_02) shows exactly how.

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
summary = json.loads(summary_path.read_text(encoding="utf-8"))
print("クラス / classes:", summary["classes"])
print("画像サイズ / image size:", summary["image_size"])
print()
splits = pd.DataFrame(summary["splits"]).T
print(splits.to_string())
print()
print("重複画像（分割間）/ duplicate images across splits:", summary["duplicate_images_across_splits"])

---

## Part 3　ラベルファイルを読む / Reading a label file

### 四角とラベルの 5 つの値 / the box and the five label values

![](../../assets/diagrams/teach_yolo_label_fields.png)

図：1 つの四角と、それに対応する YOLO 形式の 1 行（合成画像）。

Figure: one box and the single YOLO line that describes it (synthetic image).

**気づいてほしいこと**

値はすべて 0〜1 で、**画素ではなく画像に対する割合** です。だから画像を縮小してもラベルは書き換えなくて済みます。① はクラス番号で、座標ではありません。

**What to notice**

Every value is between 0 and 1 and is a **fraction of the image, not a pixel count**. That is why resizing an image does not invalidate its labels. ① is a class id, not a coordinate.

**考えてみてください / Try to answer**

- 同じ四角を 640 px の画像と 1280 px の画像で表すと、ラベルの値は変わりますか。
- If the same box sits in a 640 px and a 1280 px version of the image, do the label values change?


#### 1 行が 1 つの欠陥
ラベルファイルの 1 行は、次の 5 つの数値です。

```text
0 0.550000 0.425000 0.300000 0.250000
↑ ↑        ↑        ↑        ↑
クラス 中心x  中心y    幅      高さ        （すべて 0〜1 に正規化）
```

空のファイルは「この画像に欠陥は無い」を意味します。
**背景画像（negative sample）** として重要で、モデルが「何も無い」を学ぶのに役立ちます。

#### One line per defect
Each line has five numbers:

```text
0 0.550000 0.425000 0.300000 0.250000
^ ^        ^        ^        ^
cls centre_x centre_y  width  height    (all normalised to 0-1)
```

An empty file means "no defect in this image". Those **negative samples** matter: they teach the
model what "nothing here" looks like.

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
label_dir = dataset_dir / "labels" / "train"
label_files = sorted(label_dir.glob("*.txt"))
print("train のラベル数 / label files:", len(label_files))

empty_labels = [p for p in label_files if p.stat().st_size == 0]
print("空のラベル（背景画像）/ empty labels:", len(empty_labels))
print()

sample_label = next(p for p in label_files if p.stat().st_size > 0)
print("例 / example:", sample_label.name)
print(sample_label.read_text(encoding="utf-8"))

#### 正規化座標を戻す
ラベルの 0〜1 の値は、画像サイズを掛けるとピクセルに戻ります。

```text
x1 = (xc - w/2) * 画像の幅
y1 = (yc - h/2) * 画像の高さ
x2 = (xc + w/2) * 画像の幅
y2 = (yc + h/2) * 画像の高さ
#### Convert normalised coordinates back
```

Multiplying the normalised values by the image size returns pixels:

```text
x1 = (xc - w/2) * image_width
y1 = (yc - h/2) * image_height
x2 = (xc + w/2) * image_width
y2 = (yc + h/2) * image_height
```

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
def label_to_boxes(label_path, image_width, image_height):
    """ラベルファイルを (class_id, x1, y1, x2, y2) のリストにする。
    Convert a label file into a list of (class_id, x1, y1, x2, y2)."""
    boxes = []
    for line in label_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        class_id, xc, yc, w, h = [float(v) for v in line.split()]
        x1 = (xc - w / 2) * image_width
        y1 = (yc - h / 2) * image_height
        x2 = (xc + w / 2) * image_width
        y2 = (yc + h / 2) * image_height
        boxes.append((int(class_id), x1, y1, x2, y2))
    return boxes


image_path = dataset_dir / "images" / "train" / f"{sample_label.stem}.png"
image = cv2.imread(str(image_path))
height, width = image.shape[:2]
print("image size:", (width, height))

for box in label_to_boxes(sample_label, width, height):
    class_id, x1, y1, x2, y2 = box
    print(f"class={summary['classes'][class_id]:<10} bbox=({x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f})")

#### ラベルを画像に描いて点検する
ラベルが正しいかを確かめる最も簡単な方法は、**画像に描いてみる** ことです。
四角が欠陥を囲んでいなければ、ラベルが間違っています。

この点検は、実務でも必ず行います。**AI を学習する前に、データを疑う** のが鉄則です。

#### Draw the labels to inspect them
The simplest correctness check is to **draw the labels on the image**. If the box does
not enclose the defect, the label is wrong.

Do this every time in real work: **suspect the data before you train.**

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import random

random.seed(0)
train_images = sorted((dataset_dir / "images" / "train").glob("*.png"))
chosen = random.sample(train_images, 4)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.4))
for ax, path in zip(axes, chosen):
    img = cv2.imread(str(path))
    h, w = img.shape[:2]
    label_path = dataset_dir / "labels" / "train" / f"{path.stem}.txt"
    for class_id, x1, y1, x2, y2 in label_to_boxes(label_path, w, h):
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), 1)
        cv2.putText(img, summary["classes"][class_id], (int(x1), max(int(y1) - 4, 9)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(path.name, fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("labels drawn on the training images / 学習画像にラベルを描いたもの")
plt.tight_layout()
plt.show()

---

## Part 4　学習・検証・試験 / Train, validation and test

### 学習・検証・試験の役割 / the roles of train, validation and test

![](../../assets/diagrams/teach_split_roles.png)

図：3 つの分割の役割と、同じ撮影セッションの近い画像による情報の漏れ。

Figure: what each split is for, and how near-duplicate images from one capture session leak.

**気づいてほしいこと**

しきい値やモデルの選択に使ったデータで評価すると、結果は良く見えます。だから **試験は最後に 1 回だけ** です。ファイル名やハッシュが違っても、同じセッションで撮ったほぼ同じ絵は、分割を越えて情報を漏らします。

**What to notice**

Judging on data used to choose a threshold or a model flatters the result, so **test is used once, at the end**. Different file names and hashes do not help: near-identical pictures from one capture session leak across the split.

**考えてみてください / Try to answer**

- あなたの工程では、何を単位に分割するのが正しいですか（日付・ロット・カメラ）。
- On your line, what is the right unit to split on: date, lot, or camera?


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
show([Image.open(ASSETS / "diagrams" / "diagram_dataset_split.png")],
     ["why we split the data / データを分ける理由"], figsize=(10, 4.5))

#### 3 つの役割

| データ | 役割 | 使う場面 |
|---|---|---|
| 学習用（train） | モデルの数値を決める | 学習 |
| 検証用（val） | 学習の進み具合を確認し、設定を選ぶ | 学習中 |
| 試験用（test） | 最終的な性能を測る。**学習中は一切見ない** | 学習後、1 回だけ |

なぜ分けるか。学習用だけで測ると、モデルは **覚えてしまっている**（丸暗記）ので、
成績が良く見えます。これは **実力ではありません**。これを **過学習（overfitting）** と呼びます。

#### The three roles
| Data | Role | Used when |
|---|---|---|
| train | determines the model's numbers | training |
| val | monitors progress and selects settings | during training |
| test | measures final performance; **never seen during training** | once, afterwards |

Why split? Measuring on training data rewards **memorisation**, not skill. That is
**overfitting**.

---

## Part 5　データ漏れ / Data leakage

#### 同じ画像が複数の分割に入ると、何が起きるか
学習用と試験用に **同じ画像**（またはほぼ同じ画像）が入っていると、
モデルはその画像を **覚えているだけ** で高い成績を出します。

これは **カンニング** と同じです。実際の現場ではそんな画像は来ないので、
**現場での性能はまったく保証されません**。

これを **データ漏れ（data leakage）** と呼びます。

**漏れが起きる典型的な原因**

- 連続した動画からフレームを切り出し、隣り合うフレームを別々の分割に入れた
- 同じ製品を角度を変えて撮り、別々の分割に入れた
- データをシャッフルする前に分割した結果、偏った

この研修のデータセットは、**画像の中身（ハッシュ）で重複を検査** しています。

#### What happens when one image appears in two splits
If the **same** (or nearly the same) image appears in both train and test, the model
merely **remembers** it and scores highly.

That is cheating. Such an image will never appear in production, so **the score guarantees nothing**
on the line. This is **data leakage**.

Typical causes:

- consecutive video frames split across train and test
- the same product photographed from slightly different angles
- splitting before shuffling, producing a biased order

This course's dataset generator **checks for duplicates by image hash**.

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import hashlib

def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

seen = {}
duplicates = []
for split in ("train", "val", "test"):
    for path in sorted((dataset_dir / "images" / split).glob("*.png")):
        key = digest(path)
        if key in seen:
            duplicates.append((seen[key], str(path)))
        else:
            seen[key] = str(path)

print("検査した画像数 / images checked:", len(seen))
print("重複 / duplicates:", len(duplicates))
if duplicates:
    for first, second in duplicates[:5]:
        print("  ", first, "<->", second)
else:
    print("データ漏れはありません / no leakage detected")

---

## 演習 / Exercises

#### 演習 1（基本）：ラベルを手計算する
画像サイズ 320 × 320、欠陥の四角が `x1=100, y1=140, x2=180, y2=200` のとき、
YOLO 形式の 5 つの数値を計算してください。

#### Exercise 1 (basic): compute a label by hand
For a 320x320 image and a defect box `x1=100, y1=140, x2=180, y2=200`, compute the
five YOLO values.
Done when the line matches `0 0.437500 0.531250 0.250000 0.187500`.

In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 1 / Exercise 1
width = height = 320
x1, y1, x2, y2 = 100, 140, 180, 200

xc = 0.0
yc = 0.0
w = 0.0
h = 0.0
print(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

#### 演習 2（中級）：空のラベルを数える
train / val / test それぞれで、**空のラベル（背景画像）の割合** を計算してください。

#### Exercise 2 (intermediate): count the empty labels
For train / val / test, compute the proportion of **empty labels (background images)**.
Done when three percentages are printed with one decimal place.

In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 2 / Exercise 2
for split in ("train", "val", "test"):
    files = sorted((dataset_dir / "labels" / split).glob("*.txt"))
    empty = [p for p in files if p.stat().st_size == 0]
    ratio = 0.0   # <- 計算 / compute
    print(f"{split:<6} total={len(files):>3} empty={len(empty):>3} ratio={ratio:.1f} %")

#### 演習 3（発展）（発展・任意）：データ漏れを見つける
わざとデータ漏れを作って、検出できるか試してください。

1. `images/test` の画像を 1 枚選び、`images/train` に **コピー** する
2. 上のハッシュ検査セルを再実行する
3. 重複が検出されることを確認する
4. コピーを削除して、元に戻す

#### Exercise 3 (challenge) (challenge, optional) introduce and detect leakage
Create leakage on purpose and see whether you can detect it: copy one test image into
`images/train`, re-run the hash check, confirm the duplicate is found, then delete the copy.
Done when the checker reports exactly one duplicate and you have cleaned up.

In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 3 / Exercise 3
import hashlib
import shutil

source = sorted((dataset_dir / "images" / "test").glob("*.png"))[0]
target = dataset_dir / "images" / "train" / "leak_test.png"

# 1. わざとデータ漏れを作る / create data leakage on purpose
shutil.copyfile(source, target)
print("コピーしました / copied:", target.name)

# 2. ハッシュで検出できるか確認する / can the hash check find it?
seen = {}
leaks = 0
for split in ("train", "val", "test"):
    for path in sorted((dataset_dir / "images" / split).glob("*.png")):
        key = hashlib.sha256(path.read_bytes()).hexdigest()
        if key in seen:
            leaks += 1
            print("  重複を検出 / duplicate:", seen[key], "<->", path)
        else:
            seen[key] = str(path)
print("検出した重複 / duplicates found:", leaks)

# 3. 後片付け（必ず消す）/ clean up (always remove it)
target.unlink()
print("削除しました / removed:", target.name)
print("=> もう一度検査すると 0 件に戻ります / run the check again: back to zero")

---

#### 理解度チェック / Understanding questions

1. 教師あり学習には、何が必要ですか。2 つ挙げてください。
2. ラベルファイルの 5 つの数値は、それぞれ何を意味しますか。
3. なぜ検証データと試験データを分けるのですか。
4. データ漏れがあると、なぜ「成績が良く見えるのに現場で使えない」のですか。
5. 空のラベル（背景画像）は、なぜ必要ですか。

1. What two things does supervised learning need?
2. What do the five numbers in a label line mean?
3. Why separate validation data from test data?
4. Why does leakage give "good scores that do not work in the plant"?
5. Why are empty labels (background images) useful?
Model answers: `docs/instructor/day04_instructor_guide.md`.

---

## 困ったときは / Troubleshooting

エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状（日本語） | Symptom (English) | 対処（日本語） | What to do (English) |
| --- | --- | --- | --- |
| ラベルが無い画像がある | An image has no label file | YOLO はそれを背景画像として扱います。意図していなければ、ラベルを作る必要があります。 | YOLO treats it as a background image. If that is unintended, create the label. |
| 四角が欠陥とずれている | The box does not match the defect | アノテーションの誤りです。`label_to_boxes` で座標を確認し、ラベルを直してください。 | An annotation error. Check the coordinates and correct the label. |
| `FileNotFoundError: data.yaml` | `FileNotFoundError: data.yaml` | データセットが未生成です。`python assets\scripts\prepare_datasets.py` を実行してください。 | The dataset was never generated. Run `prepare_datasets.py`. |
| `UnicodeDecodeError` でラベルが読めない | `UnicodeDecodeError` reading a label | ラベルは ASCII のはずです。エディタで保存し直した場合は文字コードを確認してください。 | Labels should be ASCII. If you re-saved one in an editor, check its encoding. |

## 完了チェックリスト / Completion checklist

**必須の項目だけ** を並べています。**演習 3（発展）は任意** で、未実施でも完了です。時間が余ったときだけ取り組んでください。

Only the **required** items are listed. **Exercise 3 (challenge) is optional**; leaving it undone still counts as complete. Do it only if you have spare time.

すべてにチェックが付いたら、このノートブックは完了です。

This notebook is complete when every box is checked.

- [ ] Part 1〜Part 5 を読み、すべてのコードセルを実行した
      - [ ] Read Parts 1-5 and ran every code cell
- [ ] ラベルを画像に描いて点検した
      - [ ] Drew the labels on images and inspected them
- [ ] ハッシュで重複を検査した
      - [ ] Checked duplicates by hash
- [ ] 演習 1〜2（必須）を実施した
      - [ ] Completed exercises 1-2 (required)
- [ ] 理解度チェックの 5 問に答えた
      - [ ] Answered the 5 questions
- [ ] ノートブックを保存した
      - [ ] Saved the notebook

## さらに学ぶ / Further learning（任意 / optional）

### 参考リンク / Reference links

- [Ultralytics: データセット形式](https://docs.ultralytics.com/datasets/detect/)
- [Ultralytics: データ収集とアノテーション](https://docs.ultralytics.com/guides/)
- [COCO アノテーション形式](https://cocodataset.org/#format-data)

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

空のラベルを 1 枚減らして（背景画像を 1 枚欠陥画像に変えて）学習し、偽陽性の数が変わるか試してください。

Remove one background image, retrain, and see whether the number of false positives changes.